In [0]:
%sql
-- Create datacube schema if it doesn't exist
CREATE SCHEMA IF NOT EXISTS gold_catalog.datacube;

-- Create comprehensive datacube combining payroll and GL data
CREATE OR REPLACE TABLE gold_catalog.datacube.fact_integrated_datacube AS

-- Payroll Data with Dimensions
SELECT 
  'PAYROLL' AS source_fact,
  p.payroll_id AS transaction_id,
  p.pay_date AS transaction_date,
  p.pay_period_start,
  p.pay_period_end,
  p.fiscal_year,
  p.fiscal_period,
  p.fiscal_month_label,
  
  -- Employee Dimension
  e.employee_id,
  e.FullName AS employee_name,
  e.email AS employee_email,
  e.position AS employee_position,
  e.hire_date,
  e.termination_date,
  e.base_salary AS employee_base_salary,
  e.is_active AS employee_is_active,
  
  -- Department Dimension
  d.department_id,
  d.department_name,
  d.cost_center,
  
  -- Company Dimension
  c.company_id,
  c.company_name,
  c.industry,
  c.country,
  c.established_date AS company_established_date,
  c.is_active AS company_is_active,
  
  -- Account Dimension (NULL for payroll)
  CAST(NULL AS STRING) AS account_id,
  CAST(NULL AS STRING) AS account_name,
  CAST(NULL AS STRING) AS account_type,
  CAST(NULL AS STRING) AS account_category,
  
  -- Payroll Metrics
  p.gross_salary,
  p.bonus,
  p.overtime_pay,
  p.commission,
  p.allowances,
  p.total_compensation,
  p.tax_deduction,
  p.social_security,
  p.health_insurance,
  p.retirement_contribution,
  p.other_deductions,
  p.total_deductions,
  p.net_salary,
  
  -- GL Metrics (NULL for payroll)
  CAST(NULL AS DECIMAL(18,2)) AS debit_amount,
  CAST(NULL AS DECIMAL(18,2)) AS credit_amount,
  CAST(NULL AS DECIMAL(18,2)) AS net_amount,
  CAST(NULL AS STRING) AS transaction_type,
  CAST(NULL AS STRING) AS pl_section,
  CAST(NULL AS STRING) AS opex_category,
  
  p.status,
  CAST(NULL AS STRING) AS created_by,
  CAST(NULL AS STRING) AS approved_by
  
FROM gold_catalog.mart.fact_payroll p
LEFT JOIN gold_catalog.mart.dim_employee e ON p.employee_id = e.employee_id
LEFT JOIN gold_catalog.mart.dim_departments d ON p.department_id = d.department_id
LEFT JOIN gold_catalog.mart.dim_company c ON p.company_id = c.company_id

UNION ALL

-- GL Data with Dimensions
SELECT 
  'GENERAL_LEDGER' AS source_fact,
  g.gl_id AS transaction_id,
  g.posting_date AS transaction_date,
  CAST(NULL AS DATE) AS pay_period_start,
  CAST(NULL AS DATE) AS pay_period_end,
  g.fiscal_year,
  g.fiscal_period,
  g.fiscal_month_label,
  
  -- Employee Dimension (NULL for GL)
  CAST(NULL AS INT) AS employee_id,
  CAST(NULL AS STRING) AS employee_name,
  CAST(NULL AS STRING) AS employee_email,
  CAST(NULL AS STRING) AS employee_position,
  CAST(NULL AS DATE) AS hire_date,
  CAST(NULL AS DATE) AS termination_date,
  CAST(NULL AS DECIMAL(18,2)) AS employee_base_salary,
  CAST(NULL AS BOOLEAN) AS employee_is_active,
  
  -- Department Dimension
  d.department_id,
  d.department_name,
  d.cost_center,
  
  -- Company Dimension
  c.company_id,
  c.company_name,
  c.industry,
  c.country,
  c.established_date AS company_established_date,
  c.is_active AS company_is_active,
  
  -- Account Dimension
  a.account_id,
  a.account_name,
  a.account_type,
  a.category AS account_category,
  
  -- Payroll Metrics (NULL for GL)
  CAST(NULL AS DECIMAL(18,2)) AS gross_salary,
  CAST(NULL AS DECIMAL(18,2)) AS bonus,
  CAST(NULL AS DECIMAL(18,2)) AS overtime_pay,
  CAST(NULL AS DECIMAL(18,2)) AS commission,
  CAST(NULL AS DECIMAL(18,2)) AS allowances,
  CAST(NULL AS DECIMAL(18,2)) AS total_compensation,
  CAST(NULL AS DECIMAL(18,2)) AS tax_deduction,
  CAST(NULL AS DECIMAL(18,2)) AS social_security,
  CAST(NULL AS DECIMAL(18,2)) AS health_insurance,
  CAST(NULL AS DECIMAL(18,2)) AS retirement_contribution,
  CAST(NULL AS DECIMAL(18,2)) AS other_deductions,
  CAST(NULL AS DECIMAL(18,2)) AS total_deductions,
  CAST(NULL AS DECIMAL(18,2)) AS net_salary,
  
  -- GL Metrics
  g.debit_amount,
  g.credit_amount,
  g.net_amount,
  g.transaction_type,
  g.pl_section,
  g.opex_category,
  
  g.status,
  g.created_by,
  g.approved_by
  
FROM gold_catalog.mart.fact_gl g
LEFT JOIN gold_catalog.mart.dim_account a ON g.account_id = a.account_id
LEFT JOIN gold_catalog.mart.dim_departments d ON g.department_id = d.department_id
LEFT JOIN gold_catalog.mart.dim_company c ON g.company_id = c.company_id;

In [0]:
%sql
-- Verify the datacube creation
SELECT 
  source_fact,
  COUNT(*) AS total_records,
  COUNT(DISTINCT company_id) AS unique_companies,
  COUNT(DISTINCT department_id) AS unique_departments,
  MIN(transaction_date) AS earliest_transaction,
  MAX(transaction_date) AS latest_transaction
FROM gold_catalog.datacube.fact_integrated_datacube
GROUP BY source_fact
ORDER BY source_fact;

In [0]:
%sql
-- Sample records from the datacube
SELECT 
  source_fact,
  transaction_id,
  transaction_date,
  fiscal_year,
  fiscal_period,
  company_name,
  department_name,
  employee_name,
  account_name,
  COALESCE(total_compensation, net_amount) AS amount,
  status
FROM gold_catalog.datacube.fact_integrated_datacube
LIMIT 10;